In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    mean_absolute_percentage_error, explained_variance_score,
    max_error, median_absolute_error
)
from sklearn.ensemble import RandomForestRegressor
import optuna
from optuna.samplers import TPESampler
import joblib
import time
import matplotlib.pyplot as plt
import seaborn as sns

# Load the data
data = pd.read_csv('preprocessed.csv')

# Define features and target
features = ['current_stop_name', 'next_stop_name', 'day_of_week', 'is_holiday',
            'is_peak_hour', 'weather_condition', 'passenger_count', 'current_speed',
            'distance_to_next_stop', 'current_lat', 'current_lon']
target = 'eta_minutes'

X = data[features]
y = data[target]

# Convert boolean columns to int
X['is_holiday'] = X['is_holiday'].astype(int)
X['is_peak_hour'] = X['is_peak_hour'].astype(int)

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Enhanced function to evaluate model with comprehensive metrics
def evaluate_model(model, X_train, y_train, X_test, y_test, cv=5):
    # Start timer for prediction
    pred_start_time = time.time()
    y_pred = model.predict(X_test)
    prediction_time = time.time() - pred_start_time
    
    # Training predictions for overfitting analysis
    y_train_pred = model.predict(X_train)
    
    # Calculate residuals
    residuals = y_test - y_pred
    abs_errors = np.abs(residuals)
    
    # 1. Primary Regression Metrics
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    
    # Additional primary metrics
    try:
        # Handle zero values in y_test to avoid division by zero in MAPE
        valid_indices = y_test != 0
        if valid_indices.any():
            mape = mean_absolute_percentage_error(y_test[valid_indices], y_pred[valid_indices]) * 100
        else:
            mape = np.nan
    except:
        mape = np.nan
        
    evs = explained_variance_score(y_test, y_pred)
    max_err = max_error(y_test, y_pred)
    median_ae = median_absolute_error(y_test, y_pred)
    
    # 2. Cross-Validation Metrics
    try:
        cv_rmse = np.sqrt(-np.mean(cross_val_score(model, X, y, 
                                                   scoring='neg_mean_squared_error', 
                                                   cv=cv)))
        cv_mae = -np.mean(cross_val_score(model, X, y, 
                                          scoring='neg_mean_absolute_error', 
                                          cv=cv))
        cv_r2 = np.mean(cross_val_score(model, X, y, 
                                        scoring='r2', 
                                        cv=cv))
    except:
        cv_rmse, cv_mae, cv_r2 = np.nan, np.nan, np.nan
    
    # 3. Error Distribution Metrics
    p90_error = np.percentile(abs_errors, 90)
    p95_error = np.percentile(abs_errors, 95)
    p99_error = np.percentile(abs_errors, 99)
    
    # 4. Overfitting Metrics
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    overfitting_ratio = train_rmse / rmse if rmse > 0 else np.nan
    
    # 5. Residual Analysis Metrics
    residual_mean = residuals.mean()
    residual_std = residuals.std()
    
    # Print the metrics
    print("\n===== Model Performance Metrics =====")
    print("\n1. Primary Regression Metrics:")
    print(f"MAE: {mae:.4f}")
    print(f"MSE: {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R² Score: {r2:.4f}")
    print(f"MAPE: {mape:.2f}%")
    print(f"Explained Variance Score: {evs:.4f}")
    print(f"Max Error: {max_err:.4f}")
    print(f"Median Absolute Error: {median_ae:.4f}")
    
    print("\n2. Cross-Validation Metrics:")
    print(f"CV RMSE (5-fold): {cv_rmse:.4f}")
    print(f"CV MAE (5-fold): {cv_mae:.4f}")
    print(f"CV R² (5-fold): {cv_r2:.4f}")
    
    print("\n3. Error Distribution Metrics:")
    print(f"90th Percentile of Absolute Errors (P90): {p90_error:.4f}")
    print(f"95th Percentile of Absolute Errors (P95): {p95_error:.4f}")
    print(f"99th Percentile of Absolute Errors (P99): {p99_error:.4f}")
    
    print("\n4. Overfitting Analysis:")
    print(f"Training RMSE: {train_rmse:.4f}")
    print(f"Test RMSE: {rmse:.4f}")
    print(f"Overfitting Ratio (train_rmse/test_rmse): {overfitting_ratio:.4f}")
    
    print("\n5. Residual Analysis:")
    print(f"Mean of Residuals: {residual_mean:.4f}")
    print(f"Standard Deviation of Residuals: {residual_std:.4f}")
    
    print(f"\n6. Prediction Time:")
    print(f"Average prediction time: {prediction_time:.4f} seconds")
    
    # Return all metrics as a dictionary
    return {
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse,
        'R2': r2,
        'MAPE': mape,
        'EVS': evs,
        'Max_Error': max_err,
        'Median_AE': median_ae,
        'CV_RMSE': cv_rmse,
        'CV_MAE': cv_mae,
        'CV_R2': cv_r2,
        'P90_Error': p90_error,
        'P95_Error': p95_error,
        'P99_Error': p99_error,
        'Train_RMSE': train_rmse,
        'Overfitting_Ratio': overfitting_ratio,
        'Residual_Mean': residual_mean,
        'Residual_Std': residual_std,
        'Prediction_Time': prediction_time
    }

# Function to plot residual analysis
def plot_residuals(y_test, y_pred, title="Residual Analysis"):
    residuals = y_test - y_pred
    
    plt.figure(figsize=(12, 10))
    
    # Residual vs Predicted plot
    plt.subplot(2, 2, 1)
    plt.scatter(y_pred, residuals, alpha=0.5)
    plt.axhline(y=0, color='r', linestyle='-')
    plt.title('Residuals vs Predicted Values')
    plt.xlabel('Predicted Values')
    plt.ylabel('Residuals')
    
    # Residual distribution
    plt.subplot(2, 2, 2)
    sns.histplot(residuals, kde=True)
    plt.title('Distribution of Residuals')
    plt.xlabel('Residual Value')
    
    # Actual vs Predicted
    plt.subplot(2, 2, 3)
    plt.scatter(y_test, y_pred, alpha=0.5)
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
    plt.title('Actual vs Predicted')
    plt.xlabel('Actual Values')
    plt.ylabel('Predicted Values')
    
    # QQ plot
    plt.subplot(2, 2, 4)
    from scipy import stats
    stats.probplot(residuals, plot=plt)
    plt.title('Q-Q Plot of Residuals')
    
    plt.tight_layout()
    plt.suptitle(title, fontsize=16)
    plt.subplots_adjust(top=0.9)
    
    # Save the plot
    plt.savefig(f"{title.replace(' ', '_').lower()}.png")
    plt.close()

# Start timer for baseline model training
baseline_start_time = time.time()

# Baseline Random Forest model
print("Training baseline Random Forest model...")
baseline_model = RandomForestRegressor(random_state=42)
baseline_model.fit(X_train, y_train)

baseline_training_time = time.time() - baseline_start_time
print(f"Baseline model training time: {baseline_training_time:.2f} seconds")

print("\nBaseline Model Performance:")
baseline_metrics = evaluate_model(baseline_model, X_train, y_train, X_test, y_test)

# Plot residuals for baseline model
plot_residuals(y_test, baseline_model.predict(X_test), "Baseline Model Residuals")

# Optuna optimization
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'max_depth': trial.suggest_int('max_depth', 5, 30),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        'bootstrap': trial.suggest_categorical('bootstrap', [True, False]),
    }
    
    model = RandomForestRegressor(**params, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    # Calculate RMSE manually since squared=False might not be supported in all sklearn versions
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    
    return rmse

print("\nStarting Optuna optimization...")
optuna_start_time = time.time()

study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=42))
study.optimize(objective, n_trials=50)

optuna_time = time.time() - optuna_start_time
print(f"Optuna optimization time: {optuna_time:.2f} seconds")

# Train optimized model
print("\nTraining optimized model with best parameters...")
print(f"Best parameters: {study.best_params}")

optimized_start_time = time.time()

best_params = study.best_params
optimized_model = RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
optimized_model.fit(X_train, y_train)

optimized_training_time = time.time() - optimized_start_time
print(f"Optimized model training time: {optimized_training_time:.2f} seconds")

print("\nOptimized Model Performance:")
optimized_metrics = evaluate_model(optimized_model, X_train, y_train, X_test, y_test)

# Plot residuals for optimized model
plot_residuals(y_test, optimized_model.predict(X_test), "Optimized Model Residuals")

# Summarize all timing metrics
print("\n===== Timing Metrics =====")
print(f"Baseline Model Training Time: {baseline_training_time:.2f} seconds")
print(f"Optuna Optimization Time: {optuna_time:.2f} seconds")
print(f"Optimized Model Training Time: {optimized_training_time:.2f} seconds")
print(f"Total Modeling Time: {baseline_training_time + optuna_time + optimized_training_time:.2f} seconds")

# Compare baseline and optimized models with comprehensive metrics
print("\n===== Performance Comparison =====")
comparison_metrics = ['RMSE', 'MAE', 'R2', 'MAPE', 'P95_Error']

for metric in comparison_metrics:
    if metric in baseline_metrics and metric in optimized_metrics:
        baseline_value = baseline_metrics[metric]
        optimized_value = optimized_metrics[metric]
        
        if baseline_value != 0 and not np.isnan(baseline_value) and not np.isnan(optimized_value):
            if metric in ['R2', 'EVS']:  # Higher is better
                improvement = optimized_value - baseline_value
                pct_change = (improvement / abs(baseline_value)) * 100 if baseline_value != 0 else np.nan
                print(f"{metric}: Baseline = {baseline_value:.4f}, Optimized = {optimized_value:.4f}, "
                      f"Improvement: {improvement:.4f} ({pct_change:.2f}%)")
            else:  # Lower is better
                improvement = baseline_value - optimized_value
                pct_change = (improvement / baseline_value) * 100 if baseline_value != 0 else np.nan
                print(f"{metric}: Baseline = {baseline_value:.4f}, Optimized = {optimized_value:.4f}, "
                      f"Improvement: {improvement:.4f} ({pct_change:.2f}%)")
        else:
            print(f"{metric}: Baseline = {baseline_value:.4f}, Optimized = {optimized_value:.4f}")

# Save the optimized model
model_filename = 'optimized_randomforest_eta_predictor.pkl'
joblib.dump(optimized_model, model_filename)
print(f"\nOptimized model saved as {model_filename}")

# Feature importance
importance = pd.DataFrame({
    'feature': features,
    'importance': optimized_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nFeature Importance:")
print(importance)

# Plot feature importance
plt.figure(figsize=(12, 8))
sns.barplot(x='importance', y='feature', data=importance)
plt.title('Feature Importance', fontsize=16)
plt.tight_layout()
plt.savefig('feature_importance.png')
plt.close()

# Save all metrics to CSV for future reference
metrics_df = pd.DataFrame({
    'Metric': list(baseline_metrics.keys()),
    'Baseline': list(baseline_metrics.values()),
    'Optimized': list(optimized_metrics.values())
})

# Calculate improvement
metrics_df['Absolute_Improvement'] = metrics_df.apply(
    lambda row: row['Optimized'] - row['Baseline'] if row['Metric'] in ['R2', 'EVS', 'CV_R2'] 
    else row['Baseline'] - row['Optimized'], axis=1
)

metrics_df['Percentage_Improvement'] = metrics_df.apply(
    lambda row: (row['Absolute_Improvement'] / abs(row['Baseline'])) * 100 
    if row['Baseline'] != 0 and not np.isnan(row['Baseline']) else np.nan, axis=1
)

metrics_df.to_csv('model_performance_metrics.csv', index=False)
print("\nAll metrics saved to 'model_performance_metrics.csv'")

# Learning curves analysis (train size vs performance)
from sklearn.model_selection import learning_curve

def plot_learning_curve(estimator, X, y, title):
    train_sizes, train_scores, test_scores = learning_curve(
        estimator, X, y, cv=5, scoring='neg_root_mean_squared_error',
        train_sizes=np.linspace(0.1, 1.0, 10), n_jobs=-1
    )
    
    train_scores_mean = -np.mean(train_scores, axis=1)
    test_scores_mean = -np.mean(test_scores, axis=1)
    
    plt.figure(figsize=(10, 6))
    plt.plot(train_sizes, train_scores_mean, 'o-', color='r', label='Training RMSE')
    plt.plot(train_sizes, test_scores_mean, 'o-', color='g', label='Validation RMSE')
    plt.title(title, fontsize=16)
    plt.xlabel('Training Set Size', fontsize=14)
    plt.ylabel('RMSE', fontsize=14)
    plt.legend(loc='best', fontsize=12)
    plt.grid(True)
    plt.savefig(f"{title.replace(' ', '_').lower()}.png")
    plt.close()

# Create learning curve for optimized model (optional - can be commented out if taking too long)
try:
    print("\nGenerating learning curve (this might take a while)...")
    plot_learning_curve(optimized_model, X, y, "Learning Curve - Optimized Model")
    print("Learning curve generated successfully.")
except Exception as e:
    print(f"Error generating learning curve: {e}")

C:\Users\cheng\AppData\Local\Temp\ipykernel_13820\82731480.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['is_holiday'] = X['is_holiday'].astype(int)
C:\Users\cheng\AppData\Local\Temp\ipykernel_13820\82731480.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['is_peak_hour'] = X['is_peak_hour'].astype(int)


Training baseline Random Forest model...
Baseline model training time: 23.16 seconds

Baseline Model Performance:

===== Model Performance Metrics =====

1. Primary Regression Metrics:
MAE: 0.3230
MSE: 0.3108
RMSE: 0.5575
R² Score: 0.9627
MAPE: 15.29%
Explained Variance Score: 0.9627
Max Error: 6.6200
Median Absolute Error: 0.1600

2. Cross-Validation Metrics:
CV RMSE (5-fold): nan
CV MAE (5-fold): nan
CV R² (5-fold): nan

3. Error Distribution Metrics:
90th Percentile of Absolute Errors (P90): 0.8500
95th Percentile of Absolute Errors (P95): 1.0600
99th Percentile of Absolute Errors (P99): 2.0500

4. Overfitting Analysis:
Training RMSE: 0.2089
Test RMSE: 0.5575
Overfitting Ratio (train_rmse/test_rmse): 0.3746

5. Residual Analysis:
Mean of Residuals: -0.0034
Standard Deviation of Residuals: 0.5575

6. Prediction Time:
Average prediction time: 0.2866 seconds


[I 2025-04-30 18:08:05,029] A new study created in memory with name: no-name-645550d0-903e-4f2f-9e2a-2388d8b8585d



Starting Optuna optimization...


[I 2025-04-30 18:08:07,853] Trial 0 finished with value: 0.5571179307770671 and parameters: {'n_estimators': 437, 'max_depth': 29, 'min_samples_split': 15, 'min_samples_leaf': 6, 'max_features': 'sqrt', 'bootstrap': True}. Best is trial 0 with value: 0.5571179307770671.
[I 2025-04-30 18:08:10,006] Trial 1 finished with value: 0.9357313730660595 and parameters: {'n_estimators': 737, 'max_depth': 5, 'min_samples_split': 20, 'min_samples_leaf': 9, 'max_features': 'sqrt', 'bootstrap': False}. Best is trial 0 with value: 0.5571179307770671.
[I 2025-04-30 18:08:16,717] Trial 2 finished with value: 0.5363154161431836 and parameters: {'n_estimators': 489, 'max_depth': 12, 'min_samples_split': 13, 'min_samples_leaf': 2, 'max_features': None, 'bootstrap': True}. Best is trial 2 with value: 0.5363154161431836.
[I 2025-04-30 18:08:26,034] Trial 3 finished with value: 0.5393961671385145 and parameters: {'n_estimators': 563, 'max_depth': 20, 'min_samples_split': 2, 'min_samples_leaf': 7, 'max_featur

Optuna optimization time: 303.38 seconds

Training optimized model with best parameters...
Best parameters: {'n_estimators': 539, 'max_depth': 10, 'min_samples_split': 13, 'min_samples_leaf': 2, 'max_features': None, 'bootstrap': True}
Optimized model training time: 6.24 seconds

Optimized Model Performance:

===== Model Performance Metrics =====

1. Primary Regression Metrics:
MAE: 0.3142
MSE: 0.2861
RMSE: 0.5349
R² Score: 0.9656
MAPE: 14.89%
Explained Variance Score: 0.9656
Max Error: 7.3647
Median Absolute Error: 0.1556

2. Cross-Validation Metrics:
CV RMSE (5-fold): 0.5358
CV MAE (5-fold): 0.3146
CV R² (5-fold): 0.9656

3. Error Distribution Metrics:
90th Percentile of Absolute Errors (P90): 0.8156
95th Percentile of Absolute Errors (P95): 1.0329
99th Percentile of Absolute Errors (P99): 1.9883

4. Overfitting Analysis:
Training RMSE: 0.4896
Test RMSE: 0.5349
Overfitting Ratio (train_rmse/test_rmse): 0.9153

5. Residual Analysis:
Mean of Residuals: -0.0027
Standard Deviation of Res